# CelebA Leakage Experiment

This notebook evaluates target-conditioned adversarial regularization for `Attractive` prediction. Quantitative results are aggregated over five random seeds, and qualitative replay examples use the first seed.

The query vocabulary contains CelebA attributes excluding `Attractive`. A prespecified gender-associated subset is used only for the query-composition diagnostic. Policy training uses a fixed actor temperature.


In [ ]:
%load_ext autoreload
%autoreload 2
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.stats import t

from claq.analysis import (
    plot_lambda_tradeoff_summary,
    plot_rollout_comparisons,
    sample_intuition_replays,
)
from claq.config import CelebAClaqConfig, default_paths
from claq.core import (
    build_concept_dictionary,
    concept_answers_batch,
    file_sha256,
    load_answer_cache,
    load_clip_model,
    load_concept_qa_checkpoint,
    load_run_bundle,
    make_cached_answer_loader,
    make_sensitive_mask,
    save_answer_cache,
    save_bundle_checkpoint,
)
from claq.data import (
    get_celeba_concept_qa_loaders,
    get_celeba_datasets,
    get_celeba_loaders,
    get_raw_celeba_dataset,
    load_celeba_attribute_spec,
)
from claq.models import ConceptNet2
from claq.sensitive_labels import (
    build_sensitive_labels_from_concept_targets,
    load_sensitive_labels,
    save_sensitive_labels,
)
from claq.training import HistorySamplingConfig, build_claq_models, fit_concept_qa, fit_claq, seed_everything


In [ ]:
repo_root = Path.cwd().resolve()
if not (repo_root / "claq").exists() and (repo_root.parent / "claq").exists():
    repo_root = repo_root.parent

paths = default_paths(repo_root=repo_root)
paths.ensure_artifact_dirs()

config = CelebAClaqConfig()
device = config.device
SEEDS = (0, 1, 2, 3, 4)
QUALITATIVE_SEED = SEEDS[0]
seed_everything(QUALITATIVE_SEED)

figure_ext = ".svg"
download_celeba = False
experiment_name = "celeba_leakage"
qa_experiment_name = "celeba_attractive"

train_min_history = 0
train_max_history = 16
train_non_sensitive_only = False
sample_sensitive_attribute = "Male"
sensitive_target_mode = "max"
actor_eps = config.actor_eps
claq_lambda_s = 0.4

concept_qa_max_train_batches = 80 if device.type == "cpu" else None
concept_qa_max_eval_batches = 20 if device.type == "cpu" else None
claq_max_train_batches = 80 if device.type == "cpu" else None
claq_max_eval_batches = 20 if device.type == "cpu" else None
FORCE_REBUILD_ANSWER_CACHE = False
RUN_LAMBDA_TRADEOFF_SWEEP = True
CACHE_NUM_WORKERS = min(8, max(1, (os.cpu_count() or 2) // 2)) if device.type == "cuda" else 0


def figure_path(stem):
    return paths.figures_root / f"{stem}{figure_ext}"


{
    "device": str(device),
    "experiment_name": experiment_name,
    "qa_experiment_name": qa_experiment_name,
}


In [ ]:
model_clip, preprocess = load_clip_model(config.clip_model_name, device=device)
spec = load_celeba_attribute_spec(
    root=paths.data_root,
    target_attribute=config.target_attribute,
    sensitive_attributes=config.sensitive_attributes,
    download=download_celeba,
)
positive_class_idx = 1
positive_class_name = spec.class_names[positive_class_idx]
concepts = spec.concept_names
dictionary = build_concept_dictionary(model_clip=model_clip, concepts=concepts, device=device)
sens_idx = spec.sensitive_indices
sensitive_mask = make_sensitive_mask(len(concepts), sens_idx, device)
sample_sens_idx = torch.tensor([spec.query_attribute_names.index(sample_sensitive_attribute)], dtype=torch.long)

qa_train_loader, qa_valid_loader = get_celeba_concept_qa_loaders(
    transform=preprocess,
    root=paths.data_root,
    spec=spec,
    batch_size=config.batch_size,
    num_workers=config.num_workers,
    download=download_celeba,
)
claq_train_image_loader, claq_valid_image_loader, _ = get_celeba_loaders(
    transform=preprocess,
    root=paths.data_root,
    spec=spec,
    batch_size=config.batch_size,
    num_workers=CACHE_NUM_WORKERS,
    return_query_targets=True,
    download=download_celeba,
)
_, _, test_ds = get_celeba_datasets(
    transform=preprocess,
    root=paths.data_root,
    spec=spec,
    return_query_targets=False,
    download=download_celeba,
)
raw_test_ds = get_raw_celeba_dataset(paths.data_root, spec=spec, split="test", download=False)

print(f"target attribute: {spec.target_attribute}")
print(f"# queries: {len(concepts)}")
print(f"sensitive query attributes: {spec.sensitive_attribute_names}")
print(f"sample sensitive target: {sample_sensitive_attribute}")
print(f"positive class for plots: {positive_class_name}")
print(
    f"train/valid/test sizes: {len(claq_train_image_loader.dataset)}, "
    f"{len(claq_valid_image_loader.dataset)}, {len(test_ds)}"
)


In [ ]:
qa_checkpoint = paths.checkpoints_root / f"concept_qa_{qa_experiment_name}.pt"
qa_history_path = paths.runs_root / f"concept_qa_{qa_experiment_name}_history.json"

if qa_checkpoint.exists():
    answering_model = load_concept_qa_checkpoint(qa_checkpoint, device=device)
    qa_source = qa_checkpoint
else:
    qa_model = ConceptNet2().to(device)
    qa_optimizer = torch.optim.Adam(qa_model.parameters(), lr=config.learning_rate)
    qa_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        qa_optimizer,
        T_max=max(config.concept_qa_epochs, 1),
    )
    qa_history = fit_concept_qa(
        model=qa_model,
        train_loader=qa_train_loader,
        eval_loader=qa_valid_loader,
        optimizer=qa_optimizer,
        scheduler=qa_scheduler,
        num_epochs=config.concept_qa_epochs,
        model_clip=model_clip,
        dictionary=dictionary,
        class_concept_targets=None,
        clip_device=device,
        train_device=device,
        max_train_batches=concept_qa_max_train_batches,
        max_eval_batches=concept_qa_max_eval_batches,
    )
    torch.save(qa_model.state_dict(), qa_checkpoint)
    with open(qa_history_path, "w", encoding="utf-8") as handle:
        json.dump(qa_history, handle, indent=2)
    answering_model = qa_model.eval()
    qa_source = qa_checkpoint

print(f"Concept-QA ready from: {qa_source}")


In [ ]:
sensitive_labels_dir = paths.artifacts_root / "sensitive_labels" / experiment_name

label_files = [
    sensitive_labels_dir / "s_soft_train.npy",
    sensitive_labels_dir / "s_hard_train.npy",
    sensitive_labels_dir / "s_soft_test.npy",
    sensitive_labels_dir / "s_hard_test.npy",
]

if all(path.exists() for path in label_files):
    sensitive_label_cache = load_sensitive_labels(sensitive_labels_dir)
    label_source = "cache"
else:
    s_soft_train, s_hard_train = build_sensitive_labels_from_concept_targets(
        concept_targets=claq_train_image_loader.dataset.query_targets,
        sens_idx=sens_idx,
    )
    s_soft_test, s_hard_test = build_sensitive_labels_from_concept_targets(
        concept_targets=test_ds.query_targets,
        sens_idx=sens_idx,
    )
    save_sensitive_labels(
        sensitive_labels_dir,
        train_soft=s_soft_train,
        train_hard=s_hard_train,
        test_soft=s_soft_test,
        test_hard=s_hard_test,
    )
    sensitive_label_cache = load_sensitive_labels(sensitive_labels_dir)
    label_source = "built_and_saved"

print(f"Query-sensitive labels ready from: {label_source} -> {sensitive_labels_dir}")
print(
    {
        "s_soft_train": sensitive_label_cache["s_soft_train"].shape,
        "s_hard_train": sensitive_label_cache["s_hard_train"].shape,
        "s_soft_test": sensitive_label_cache["s_soft_test"].shape,
        "s_hard_test": sensitive_label_cache["s_hard_test"].shape,
    }
)
print(
    "Mean query-sensitive activation (train/test):",
    float(sensitive_label_cache["s_soft_train"].mean()),
    float(sensitive_label_cache["s_soft_test"].mean()),
)
print(f"Lambda sample-sensitive target: {sample_sensitive_attribute}")

answer_cache_dir = paths.artifacts_root / "concept_answers" / qa_experiment_name
answer_cache_metadata = {
    "dataset": "celeba",
    "target_attribute": spec.target_attribute,
    "concept_count": len(concepts),
    "qa_checkpoint": qa_checkpoint.name,
    "qa_checkpoint_sha256": file_sha256(qa_checkpoint),
    "threshold": config.threshold_for_binarization,
    "sensitive_target": sample_sensitive_attribute,
}
answer_cache_paths = {
    "train": answer_cache_dir / "train_hard_answers.pt",
    "validation": answer_cache_dir / "validation_hard_answers.pt",
}


@torch.no_grad()
def build_answer_cache(loader, path):
    answers_parts, labels_parts, sensitive_parts = [], [], []
    for images, labels, concept_targets in loader:
        answers_parts.append(
            concept_answers_batch(
                images=images,
                model_clip=model_clip,
                dictionary=dictionary,
                answering_model=answering_model,
                clip_device=device,
                train_device=device,
                threshold=config.threshold_for_binarization,
            ).cpu()
        )
        labels_parts.append(labels.cpu())
        sensitive_parts.append(
            concept_targets[:, sample_sens_idx.cpu()].amax(dim=1).float().cpu()
        )
    save_answer_cache(
        path,
        answers=torch.cat(answers_parts),
        labels=torch.cat(labels_parts),
        sensitive_targets=torch.cat(sensitive_parts),
        metadata=answer_cache_metadata,
    )


for split, image_loader in {
    "train": claq_train_image_loader,
    "validation": claq_valid_image_loader,
}.items():
    cache_path = answer_cache_paths[split]
    if FORCE_REBUILD_ANSWER_CACHE or not cache_path.exists():
        ordered_loader = torch.utils.data.DataLoader(
            image_loader.dataset,
            batch_size=config.batch_size,
            shuffle=False,
            num_workers=CACHE_NUM_WORKERS,
            pin_memory=device.type == "cuda",
            persistent_workers=CACHE_NUM_WORKERS > 0,
        )
        build_answer_cache(ordered_loader, cache_path)

train_answer_cache = load_answer_cache(
    answer_cache_paths["train"], expected_metadata=answer_cache_metadata
)
validation_answer_cache = load_answer_cache(
    answer_cache_paths["validation"], expected_metadata=answer_cache_metadata
)
claq_train_loader = make_cached_answer_loader(
    train_answer_cache,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=device.type == "cuda",
)
claq_valid_loader = make_cached_answer_loader(
    validation_answer_cache,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=device.type == "cuda",
)
print({"answer_cache": "ready", "train_rows": len(claq_train_loader.dataset)})


In [ ]:
def load_or_train_bundle(
    run_name,
    lambda_s,
    seed,
    min_history=train_min_history,
    max_history=train_max_history,
    non_sensitive_only=train_non_sensitive_only,
    epochs=config.default_train_epochs,
    learning_rate=config.learning_rate,
    max_train_batches=claq_max_train_batches,
    max_eval_batches=claq_max_eval_batches,
    force_retrain=False,
):
    seed_everything(seed)
    run_stem = f"{experiment_name}_{run_name}_seed_{seed}"
    ckpt_path = paths.checkpoints_root / f"{run_stem}_best.pt"
    history_path = paths.runs_root / f"{run_stem}_history.json"
    if ckpt_path.exists() and history_path.exists() and not force_retrain:
        bundle = load_run_bundle(
            ckpt_path,
            device=device,
            max_queries=len(concepts),
            num_classes=config.num_classes,
            actor_eps=actor_eps,
        )
        bundle.update({"run_name": run_name, "lambda_s": lambda_s, "seed": seed})
        return bundle

    actor, classifier, s_head = build_claq_models(
        max_queries=len(concepts),
        num_classes=config.num_classes,
        device=device,
        actor_eps=config.actor_eps,
    )
    optimizer = torch.optim.Adam(
        list(actor.parameters()) + list(classifier.parameters()) + list(s_head.parameters()),
        lr=learning_rate,
    )
    history_config = HistorySamplingConfig(
        min_history=min_history,
        max_history=max_history,
        non_sensitive_only=non_sensitive_only,
    )
    history, best = fit_claq(
        actor=actor,
        classifier=classifier,
        s_head=s_head,
        optimizer=optimizer,
        train_loader=claq_train_loader,
        test_loader=claq_valid_loader,
        model_clip=model_clip,
        dictionary=dictionary,
        answering_model=answering_model,
        sens_idx=sens_idx,
        history_config=history_config,
        clip_device=device,
        train_device=device,
        threshold_for_binarization=config.threshold_for_binarization,
        lambda_s=lambda_s,
        lambda_c=0.0,
        sensitive_tau=config.sensitive_tau,
        sensitive_topk=config.sensitive_topk,
        num_epochs=epochs,
        max_train_batches=max_train_batches,
        max_test_batches=max_eval_batches,
        sensitive_target_mode=sensitive_target_mode,
        sensitive_target_indices=sample_sens_idx,
    )
    actor.load_state_dict(best["actor_state_dict"])
    classifier.load_state_dict(best["classifier_state_dict"])
    s_head.load_state_dict(best["s_head_state_dict"])
    save_bundle_checkpoint(
        checkpoint_path=ckpt_path,
        actor=actor,
        classifier=classifier,
        s_head=s_head,
        metadata={
            "run_name": run_name,
            "lambda_s": lambda_s,
            "seed": seed,
            "target_attribute": spec.target_attribute,
            "sample_sensitive_attribute": sample_sensitive_attribute,
            "query_sensitive_attributes": spec.sensitive_attribute_names,
            "sensitive_target_mode": sensitive_target_mode,
            "sensitive_conditioning": "conditional_y",
            "best_test_acc": best["test_acc"],
            "best_epoch": best["epoch"],
            "history_config": {
                "min_history": history_config.min_history,
                "max_history": history_config.max_history,
                "non_sensitive_only": history_config.non_sensitive_only,
            },
            "actor_eps": actor_eps,
        },
    )
    history_with_seed = [{**row, "run_name": run_name, "lambda_s": lambda_s, "seed": seed} for row in history]
    with open(history_path, "w", encoding="utf-8") as handle:
        json.dump(history_with_seed, handle, indent=2)
    bundle = load_run_bundle(
        ckpt_path,
        device=device,
        max_queries=len(concepts),
        num_classes=config.num_classes,
        actor_eps=actor_eps,
    )
    bundle.update({"run_name": run_name, "lambda_s": lambda_s, "seed": seed})
    return bundle


primary_bundles_by_seed = {
    seed: {
        "baseline": load_or_train_bundle("baseline", lambda_s=0.0, seed=seed),
        "claq": load_or_train_bundle("lam_0.40", lambda_s=claq_lambda_s, seed=seed),
    }
    for seed in SEEDS
}
baseline_bundle = primary_bundles_by_seed[QUALITATIVE_SEED]["baseline"]
claq_bundle = primary_bundles_by_seed[QUALITATIVE_SEED]["claq"]

print(f"Qualitative seed: {QUALITATIVE_SEED}")
print(baseline_bundle["ckpt_path"])
print(claq_bundle["ckpt_path"])


def answer_builder(images):
    return concept_answers_batch(
        images=images,
        model_clip=model_clip,
        dictionary=dictionary,
        answering_model=answering_model,
        clip_device=device,
        train_device=device,
        threshold=config.threshold_for_binarization,
    )


In [ ]:
intuition_records = sample_intuition_replays(
    dataset=test_ds,
    answer_builder=answer_builder,
    baseline_bundle=baseline_bundle,
    claq_bundle=claq_bundle,
    concepts=concepts,
    sensitive_mask=sensitive_mask,
    class_names=spec.class_names,
    num_cases=6,
    pool_size=500 if device.type == "cpu" else 1500,
    prefer_baseline_sensitive=False,
    confidence_threshold=config.confidence_threshold,
    rollout_max_steps=8,
    positive_class_idx=positive_class_idx,
    positive_class_name=positive_class_name,
    balance_labels=True,
    balance_concept_idx=int(sample_sens_idx.item()),
    balance_concept_name=sample_sensitive_attribute,
    random_seed=QUALITATIVE_SEED,
)

intuition_fig = plot_rollout_comparisons(
    records=intuition_records,
    raw_dataset=raw_test_ds,
    output_path=figure_path(f"{experiment_name}_rollout_replay_examples"),
    title_prefix="rollout replay",
)

print(f"Saved rollout replay figure: {intuition_fig}")


## Lambda Trade-Off Sweep

Train or load each lambda condition for five seeds using a fixed actor temperature. The final epoch is summarized per seed, then aggregated with 95% confidence intervals across seeds.


In [ ]:
lambda_sweep_values = (
    (0.0, 0.2, 0.4) if RUN_LAMBDA_TRADEOFF_SWEEP else (0.0, 0.4)
)


def lambda_run_name(lambda_s):
    return "baseline" if lambda_s == 0.0 else f"lam_{lambda_s:.2f}"


sweep_bundles_by_seed = {}
for seed in SEEDS:
    sweep_bundles_by_seed[seed] = {
        "baseline": primary_bundles_by_seed[seed]["baseline"],
        "lam_0.40": primary_bundles_by_seed[seed]["claq"],
    }
    if RUN_LAMBDA_TRADEOFF_SWEEP:
        sweep_bundles_by_seed[seed]["lam_0.20"] = load_or_train_bundle(
            "lam_0.20", lambda_s=0.2, seed=seed
        )


def load_sweep_history(run_name, seed):
    history_path = paths.runs_root / f"{experiment_name}_{run_name}_seed_{seed}_history.json"
    with open(history_path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def summarize_sweep_run(lambda_s, seed):
    run_name = lambda_run_name(lambda_s)
    history = load_sweep_history(run_name, seed)
    final_row = max(history, key=lambda row: row["epoch"])
    return {
        "run_name": run_name,
        "lambda_s": lambda_s,
        "seed": seed,
        "epoch": final_row["epoch"],
        "validation_acc": final_row["test_acc"],
        "validation_sens_q_rate": final_row["test_sens_q_rate"],
        "validation_loss": final_row["test_loss"],
        "validation_q_entropy": final_row["test_q_entropy"],
        "actor_eps": final_row["actor_eps"],
    }


lambda_sweep_by_seed = pd.DataFrame(
    [
        summarize_sweep_run(lambda_s, seed)
        for seed in SEEDS
        for lambda_s in lambda_sweep_values
    ]
)

summary = (
    lambda_sweep_by_seed.groupby(["run_name", "lambda_s"], as_index=False)
    .agg(
        seeds=("seed", "nunique"),
        epoch=("epoch", "mean"),
        validation_acc=("validation_acc", "mean"),
        validation_acc_std=("validation_acc", "std"),
        validation_sens_q_rate=("validation_sens_q_rate", "mean"),
        validation_sens_q_rate_std=("validation_sens_q_rate", "std"),
        validation_loss=("validation_loss", "mean"),
        validation_loss_std=("validation_loss", "std"),
        validation_q_entropy=("validation_q_entropy", "mean"),
        actor_eps=("actor_eps", "mean"),
    )
    .sort_values("lambda_s")
    .reset_index(drop=True)
)
t_critical = float(t.ppf(0.975, len(SEEDS) - 1))
summary["validation_acc_ci95"] = (
    t_critical * summary["validation_acc_std"] / np.sqrt(summary["seeds"])
)
summary["validation_sens_q_rate_ci95"] = (
    t_critical * summary["validation_sens_q_rate_std"] / np.sqrt(summary["seeds"])
)

lambda_sweep_rows = summary.to_dict(orient="records")
lambda_sweep_path = paths.runs_root / f"{experiment_name}_lambda_tradeoff_summary.json"
lambda_sweep_by_seed_path = paths.runs_root / f"{experiment_name}_lambda_tradeoff_by_seed.csv"
summary_path = paths.runs_root / f"{experiment_name}_lambda_tradeoff_summary.csv"
with open(lambda_sweep_path, "w", encoding="utf-8") as handle:
    json.dump(lambda_sweep_rows, handle, indent=2)
lambda_sweep_by_seed.to_csv(lambda_sweep_by_seed_path, index=False)
summary.to_csv(summary_path, index=False)

display(summary)


In [ ]:
lambda_tradeoff_figure = plot_lambda_tradeoff_summary(
    lambda_sweep_rows,
    output_path=figure_path(f"{experiment_name}_lambda_tradeoff"),
)

print(f"Saved lambda sweep summary: {lambda_sweep_path}")
print(f"Saved lambda trade-off figure: {lambda_tradeoff_figure}")
